In [1]:
# STEP 1: Imports and environment setup
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, KFold, cross_validate, cross_val_predict
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
MODEL_DIR = ROOT / 'model'
TEST_SIZE = 0.2
print('Environment ready. DATA_DIR=', DATA_DIR, ' MODEL_DIR=', MODEL_DIR)


Environment ready. DATA_DIR= /Users/kanishk/Desktop/survey analysis/data  MODEL_DIR= /Users/kanishk/Desktop/survey analysis/model


# STEP 2: Load data

In [2]:
def find_csv(path: Path):
    files = list(path.glob('*.csv'))
    return files[0] if files else None

def load_data():
    csv = find_csv(DATA_DIR)
    if csv is None:
        raise FileNotFoundError(f'No CSV found in {DATA_DIR}')
    print(f'Loading {csv}')
    return pd.read_csv(csv)

df = load_data()
df.head()

Loading /Users/kanishk/Desktop/survey analysis/data/survey data.csv


,Respondent_ID,Age_Group,Gender,Ward_Number_Zone,Survey_Topic,Citizen_Feedback_Text,State,City,Area/locality
0,RES_1001,18-25,Prefer not to say,Ward 21 - Bengaluru Zone,Electricity/Power,"Street light poles have exposed wires, very da...",Karnataka,Bengaluru,Indiranagar
1,RES_1002,26-40,Female,Ward 20 - New Delhi Zone,Water Supply,Irregular timing of water supply in the colony.,Delhi,New Delhi,Connaught Place
2,RES_1003,26-40,Male,Ward 6 - New Delhi Zone,Electricity/Power,Sparking observed in the transformer near the ...,Delhi,New Delhi,Karol Bagh
3,RES_1004,18-25,Male,Ward 24 - New Delhi Zone,Waste Management,Need more segregated waste bins in the residen...,Delhi,New Delhi,Connaught Place
4,RES_1005,18-25,Prefer not to say,Ward 14 - Bengaluru Zone,Public Health/Hospitals,Need more mosquito fogging drives in the area.,Karnataka,Bengaluru,Whitefield


# STEP 3: Basic cleaning

In [3]:
# safe basic cleaning: convert date columns and ensure Length_of_Stay exists
if 'Date of Admission' in df.columns:
    df['Date of Admission'] = pd.to_datetime(df['Date of Admission'], errors='coerce')
if 'Discharge Date' in df.columns:
    df['Discharge Date'] = pd.to_datetime(df['Discharge Date'], errors='coerce')
if 'Date of Admission' in df.columns and 'Discharge Date' in df.columns:
    df['Length_of_Stay'] = (df['Discharge Date'] - df['Date of Admission']).dt.days
    median_los = int(df['Length_of_Stay'].dropna().loc[lambda s: s >= 0].median() if not df['Length_of_Stay'].dropna().empty else 3)
    df['Length_of_Stay'] = df['Length_of_Stay'].apply(lambda x: median_los if (pd.isna(x) or x < 0) else x)
else:
    df['Length_of_Stay'] = df.get('Length_of_Stay', 3)

print('Basic cleaning done.')
df.head()

Basic cleaning done.


,Respondent_ID,Age_Group,Gender,Ward_Number_Zone,Survey_Topic,Citizen_Feedback_Text,State,City,Area/locality,Length_of_Stay
0,RES_1001,18-25,Prefer not to say,Ward 21 - Bengaluru Zone,Electricity/Power,"Street light poles have exposed wires, very da...",Karnataka,Bengaluru,Indiranagar,3
1,RES_1002,26-40,Female,Ward 20 - New Delhi Zone,Water Supply,Irregular timing of water supply in the colony.,Delhi,New Delhi,Connaught Place,3
2,RES_1003,26-40,Male,Ward 6 - New Delhi Zone,Electricity/Power,Sparking observed in the transformer near the ...,Delhi,New Delhi,Karol Bagh,3
3,RES_1004,18-25,Male,Ward 24 - New Delhi Zone,Waste Management,Need more segregated waste bins in the residen...,Delhi,New Delhi,Connaught Place,3
4,RES_1005,18-25,Prefer not to say,Ward 14 - Bengaluru Zone,Public Health/Hospitals,Need more mosquito fogging drives in the area.,Karnataka,Bengaluru,Whitefield,3


# STEP 4: Prepare pipeline

In [ ]:
def prepare_pipeline(df: pd.DataFrame):
    df = df.copy()
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if write_app:
    with open('app.py','w') as f:
        f.write(streamlit_app_code)
    print('Wrote app.py')
else:
    print('Streamlit app code ready. Set write_app = True to export app.py')

In [9]:
# STEP 1: Imports and environment setup
import os
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import json
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
MODEL_DIR = ROOT / 'model'
TEST_SIZE = 0.2

print('Environment ready. DATA_DIR=', DATA_DIR, ' MODEL_DIR=', MODEL_DIR)


Environment ready. DATA_DIR= /Users/kanishk/Desktop/survey analysis/data  MODEL_DIR= /Users/kanishk/Desktop/survey analysis/model


# STEP 4: Load the Data

In [10]:
def find_csv(path: Path):
    files = list(path.glob('*.csv'))
    return files[0] if files else None

def load_data():
    csv = find_csv(DATA_DIR)
    if csv is None:
        raise FileNotFoundError(f'No CSV found in {DATA_DIR}')
    print(f'Loading {csv}')
    return pd.read_csv(csv)

df = load_data()
df.head()

Loading /Users/kanishk/Desktop/survey analysis/data/survey data.csv


,Respondent_ID,Age_Group,Gender,Ward_Number_Zone,Survey_Topic,Citizen_Feedback_Text,State,City,Area/locality
0,RES_1001,18-25,Prefer not to say,Ward 21 - Bengaluru Zone,Electricity/Power,"Street light poles have exposed wires, very da...",Karnataka,Bengaluru,Indiranagar
1,RES_1002,26-40,Female,Ward 20 - New Delhi Zone,Water Supply,Irregular timing of water supply in the colony.,Delhi,New Delhi,Connaught Place
2,RES_1003,26-40,Male,Ward 6 - New Delhi Zone,Electricity/Power,Sparking observed in the transformer near the ...,Delhi,New Delhi,Karol Bagh
3,RES_1004,18-25,Male,Ward 24 - New Delhi Zone,Waste Management,Need more segregated waste bins in the residen...,Delhi,New Delhi,Connaught Place
4,RES_1005,18-25,Prefer not to say,Ward 14 - Bengaluru Zone,Public Health/Hospitals,Need more mosquito fogging drives in the area.,Karnataka,Bengaluru,Whitefield


In [11]:
# STEP 5: Basic cleaning
# Convert date columns and compute Length_of_Stay if possible
if 'Date of Admission' in df.columns:
    df['Date of Admission'] = pd.to_datetime(df['Date of Admission'], errors='coerce')
if 'Discharge Date' in df.columns:
    df['Discharge Date'] = pd.to_datetime(df['Discharge Date'], errors='coerce')
if 'Date of Admission' in df.columns and 'Discharge Date' in df.columns:
    df['Length_of_Stay'] = (df['Discharge Date'] - df['Date of Admission']).dt.days
    median_los = int(df['Length_of_Stay'].dropna().loc[lambda s: s >= 0].median() if not df['Length_of_Stay'].dropna().empty else 3)
    df['Length_of_Stay'] = df['Length_of_Stay'].apply(lambda x: median_los if (pd.isna(x) or x < 0) else x)
else:
    df['Length_of_Stay'] = df.get('Length_of_Stay', 3)

print('Basic cleaning done.')
df.head()

Basic cleaning done.


,Respondent_ID,Age_Group,Gender,Ward_Number_Zone,Survey_Topic,Citizen_Feedback_Text,State,City,Area/locality,Length_of_Stay
0,RES_1001,18-25,Prefer not to say,Ward 21 - Bengaluru Zone,Electricity/Power,"Street light poles have exposed wires, very da...",Karnataka,Bengaluru,Indiranagar,3
1,RES_1002,26-40,Female,Ward 20 - New Delhi Zone,Water Supply,Irregular timing of water supply in the colony.,Delhi,New Delhi,Connaught Place,3
2,RES_1003,26-40,Male,Ward 6 - New Delhi Zone,Electricity/Power,Sparking observed in the transformer near the ...,Delhi,New Delhi,Karol Bagh,3
3,RES_1004,18-25,Male,Ward 24 - New Delhi Zone,Waste Management,Need more segregated waste bins in the residen...,Delhi,New Delhi,Connaught Place,3
4,RES_1005,18-25,Prefer not to say,Ward 14 - Bengaluru Zone,Public Health/Hospitals,Need more mosquito fogging drives in the area.,Karnataka,Bengaluru,Whitefield,3


# STEP 6: Quick EDA (Exploratory Data Analysis)

In [12]:
# Basic summaries
display(df.info())
display(df.describe(include='all'))

# Categorical summary (top values)
cat_cols = df.select_dtypes(include=[object, 'category']).columns.tolist()
{c: df[c].value_counts().head(10).to_dict() for c in cat_cols}

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Respondent_ID          200 non-null    object
 1   Age_Group              200 non-null    object
 2   Gender                 200 non-null    object
 3   Ward_Number_Zone       200 non-null    object
 4   Survey_Topic           200 non-null    object
 5   Citizen_Feedback_Text  200 non-null    object
 6   State                  200 non-null    object
 7   City                   200 non-null    object
 8   Area/locality          200 non-null    object
 9   Length_of_Stay         200 non-null    int64 
dtypes: int64(1), object(9)
memory usage: 15.8+ KB


None

,Respondent_ID,Age_Group,Gender,Ward_Number_Zone,Survey_Topic,Citizen_Feedback_Text,State,City,Area/locality,Length_of_Stay
count,200,200,200,200,200,200,200,200,200,200.0
unique,200,4,4,85,5,20,3,5,7,NaN
top,RES_1001,41-60,Non-binary,Ward 19 - Bengaluru Zone,Waste Management,Need more segregated waste bins in the residen...,Karnataka,Bengaluru,Indiranagar,NaN
freq,1,58,54,8,45,17,82,82,45,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0


{'Respondent_ID': {'RES_1001': 1,
  'RES_1138': 1,
  'RES_1128': 1,
  'RES_1129': 1,
  'RES_1130': 1,
  'RES_1131': 1,
  'RES_1132': 1,
  'RES_1133': 1,
  'RES_1134': 1,
  'RES_1135': 1},
 'Age_Group': {'41-60': 58, '18-25': 50, '60+': 48, '26-40': 44},
 'Gender': {'Non-binary': 54,
  'Female': 49,
  'Male': 49,
  'Prefer not to say': 48},
 'Ward_Number_Zone': {'Ward 19 - Bengaluru Zone': 8,
  'Ward 11 - Bengaluru Zone': 8,
  'Ward 21 - Bengaluru Zone': 7,
  'Ward 13 - Bengaluru Zone': 6,
  'Ward 25 - New Delhi Zone': 5,
  'Ward 15 - Bengaluru Zone': 5,
  'Ward 20 - Bengaluru Zone': 5,
  'Ward 16 - Bengaluru Zone': 4,
  'Ward 9 - New Delhi Zone': 4,
  'Ward 15 - New Delhi Zone': 4},
 'Survey_Topic': {'Waste Management': 45,
  'Road Infrastructure': 42,
  'Public Health/Hospitals': 39,
  'Electricity/Power': 37,
  'Water Supply': 37},
 'Citizen_Feedback_Text': {'Need more segregated waste bins in the residential area.': 17,
  'Potholes on the main road are causing daily traffic jams and

In [13]:
# STEP 15: Prediction helpers
# Small utility to predict a single row (used by Streamlit app)
def predict_row(row: dict):
    model_clf = load_model('sentiment_classifier.pkl')
    model_reg = load_model('satisfaction_regressor.pkl')
    if model_clf is None and model_reg is None:
        raise RuntimeError('No models available')
    df_row = pd.DataFrame([row])
    preproc, features, targets = prepare_pipeline(df)
    # align columns: add missing feature cols with NA
    for c in features:
        if c not in df_row.columns:
            df_row[c] = pd.NA
    X_row = df_row[features]
    out = {}
    if model_clf is not None:
        out['pred_sentiment'] = model_clf.predict(X_row)[0]
    if model_reg is not None:
        out['pred_satisfaction'] = float(model_reg.predict(X_row)[0])
    return out


In [14]:
def prepare_pipeline(df: pd.DataFrame):
    df = df.copy()
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df.select_dtypes(include=[object, "category"]).columns.tolist()

    # identify targets by common names; adjust if your column names differ
    targets = [c for c in df.columns if c.lower() in ("satisfaction_score", "ai_predicted_sentiment", "urgency_level")]
    features = [c for c in df.columns if c not in targets]

    numeric_features = [c for c in features if c in numeric_cols]
    categorical_features = [c for c in features if c in cat_cols]

    numeric_transform = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    # Robust OneHotEncoder creation for different sklearn versions
    try:
        onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)
    except TypeError:
        onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

    categorical_transform = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", onehot),
    ])

    transformers = []
    if numeric_features:
        transformers.append(("num", numeric_transform, numeric_features))
    if categorical_features:
        transformers.append(("cat", categorical_transform, categorical_features))

    if not transformers:
        # no features to transform; create passthrough identity transformer
        preproc = ColumnTransformer([], remainder="drop")
    else:
        preproc = ColumnTransformer(transformers, remainder="drop")

    return preproc, features, targets

# STEP 9: Helper — Train and Save Models
Trains a Random Forest classifier for `AI_Predicted_Sentiment` (if present) and a Random Forest regressor for `Satisfaction_Score` (if present). Saves models to `model/` and a `training_report.json`.

In [15]:
def train_and_save(df: pd.DataFrame, test_size=0.2):
    preproc, features, targets = prepare_pipeline(df)
    X = df[features]
    results = {}

    if 'Satisfaction_Score' in df.columns:
        y_reg = df['Satisfaction_Score']
        X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=test_size, random_state=42)
        reg = Pipeline([('preproc', preproc), ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))])
        print('Training regressor...')
        reg.fit(X_train, y_train)
        preds = reg.predict(X_test)
        rmse = mean_squared_error(y_test, preds, squared=False)
        print(f'Regressor RMSE: {rmse:.4f}')
        joblib.dump(reg, MODEL_DIR / 'satisfaction_regressor.pkl')
        results['regression'] = {'rmse': float(rmse)}

    if 'AI_Predicted_Sentiment' in df.columns:
        y_clf = df['AI_Predicted_Sentiment'].astype(str)
        X_train, X_test, y_train, y_test = train_test_split(X, y_clf, test_size=test_size, stratify=y_clf, random_state=42)
        clf = Pipeline([('preproc', preproc), ('model', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))])
        print('Training classifier...')
        clf.fit(X_train, y_train)
        preds = clf.predict(X_test)
        acc = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds, average='weighted', zero_division=0)
        rec = recall_score(y_test, preds, average='weighted', zero_division=0)
        f1 = f1_score(y_test, preds, average='weighted', zero_division=0)
        print(f'Classifier Acc: {acc:.4f}, F1: {f1:.4f}')
        joblib.dump(clf, MODEL_DIR / 'sentiment_classifier.pkl')
        results['classification'] = {'accuracy': float(acc), 'precision': float(prec), 'recall': float(rec), 'f1': float(f1)}

    # save a report
    fi = {}
    for name in ('sentiment_classifier.pkl', 'satisfaction_regressor.pkl'):
        path = MODEL_DIR / name
        if path.exists():
            model = joblib.load(path)
            try:
                rf = model.named_steps.get('model') if hasattr(model, 'named_steps') else model
                importances = rf.feature_importances_
                fi[name] = importances.tolist()
            except Exception:
                fi[name] = None

    report = {'results': results, 'feature_importances': fi}
    with open(MODEL_DIR / 'training_report.json', 'w') as f:
        json.dump(report, f, indent=2)
    print('Saved models and training_report.json')
    return report

# To execute training, uncomment and run: report = train_and_save(df)


In [16]:
# STEP 10: Run training (execute this cell to train models and save to model/)
# Note: training can take several minutes depending on dataset size.
TEST_SIZE = 0.2  # 0.2 -> 80/20 train/test split
confirm = True  # set False to skip accidental runs
if confirm:
    report = train_and_save(df, test_size=TEST_SIZE)
    print('Training report:')
    print(report)
    print('Models saved to', MODEL_DIR)
else:
    print('Training skipped. Set confirm = True to run training.')


Saved models and training_report.json
Training report:
{'results': {}, 'feature_importances': {'sentiment_classifier.pkl': [0.0031796933857183513, 9.917716291238741e-05, 0.00014936490346326408, 0.0003400638267947745, 0.0004403317897123009, 1.1446886446886452e-05, 0.0009197400254130508, 0.003298986368324916, 1.1646866992778948e-05, 4.8799105868421124e-05, 0.0024469399976481438, 0.0004945591701041276, 0.00011736424730789978, 0.0034998082744235963, 0.0029812883262248084, 7.4592074592074415e-06, 0.0030483760441552105, 0.000965269400693737, 0.0017743311271630857, 0.000598210112949475, 0.0026720440677504686, 7.030510097645444e-05, 0.0025052111349484867, 9.7917315702218e-05, 0.00020157737688705497, 0.00025092217276799495, 1.0895191688086157e-05, 0.00015581678939543774, 0.00028025830133723227, 0.0003635526755856728, 0.00023217709457471863, 0.002729771357696858, 0.0002002451273764638, 7.318946713494992e-05, 0.001699279787681788, 0.002822913254322718, 0.000186839697096777, 0.0001922758107287631,

/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.6.1 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.6.1 when using version 1.5.1. This might lead to breaking cod

In [17]:
# STEP 11: Regressor evaluation — metrics and plots
# Shows RMSE/MAE/R2, a sample table, prediction vs truth scatter and residuals histogram.
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

try:
    TEST_SIZE
except NameError:
    TEST_SIZE = 0.2

if 'Satisfaction_Score' in df.columns:
    print('\n=== Regressor evaluation (Satisfaction_Score) ===')
    preproc, features, targets = prepare_pipeline(df)
    X = df[features]
    y_reg = df['Satisfaction_Score']
    X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=TEST_SIZE, random_state=42)
    reg_model = load_model('satisfaction_regressor.pkl')
    if reg_model is None:
        print('Regressor not found (satisfaction_regressor.pkl)')
        files = list(MODEL_DIR.glob('*.pkl'))
        if files:
            print('Model files present:')
            for p in files:
                print(' -', p.name)
    else:
        preds = reg_model.predict(X_test)
        y_true = pd.Series(y_test).reset_index(drop=True)
        y_pred = pd.Series(preds).reset_index(drop=True)
        df_eval = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred})

        rmse = mean_squared_error(y_true, y_pred, squared=False)
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        print(f'RMSE: {rmse:.4f}  |  MAE: {mae:.4f}  |  R2: {r2:.4f}')

        print('\nSample of true vs predicted (first 10 rows):')
        display(df_eval.head(10))

        # Prediction vs Truth scatter
        plt.figure(figsize=(6,6))
        sns.scatterplot(x='y_true', y='y_pred', data=df_eval, alpha=0.7)
        maxv = max(df_eval[['y_true','y_pred']].max().max(), 1)
        plt.plot([0,maxv],[0,maxv], '--', color='gray')
        plt.xlabel('True Satisfaction')
        plt.ylabel('Predicted Satisfaction')
        plt.title('Prediction vs Truth')
        plt.tight_layout()
        plt.show()

        # Residuals histogram
        residuals = y_true - y_pred
        plt.figure(figsize=(6,4))
        sns.histplot(residuals, bins=30, kde=True)
        plt.title('Residuals (y_true - y_pred)')
        plt.xlabel('Residual')
        plt.tight_layout()
        plt.show()

else:
    print('Column `Satisfaction_Score` not found in dataframe; skipping regressor evaluation.')


Column `Satisfaction_Score` not found in dataframe; skipping regressor evaluation.


# STEP 11: Prediction helpers

In [18]:
# STEP 12: Classifier evaluation — metrics, classification report and confusion heatmap
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

if 'AI_Predicted_Sentiment' in df.columns:
    print('\n=== Classifier evaluation (AI_Predicted_Sentiment) ===')
    preproc, features, targets = prepare_pipeline(df)
    X = df[features]
    y_clf = df['AI_Predicted_Sentiment'].astype(str)
    X_train, X_test, y_train, y_test = train_test_split(X, y_clf, test_size=TEST_SIZE, stratify=y_clf, random_state=42)
    clf_model = load_model('sentiment_classifier.pkl')
    if clf_model is None:
        print('Classifier not found (sentiment_classifier.pkl)')
        files = list(MODEL_DIR.glob('*.pkl'))
        if files:
            print('Model files present:')
            for p in files:
                print(' -', p.name)
    else:
        preds = clf_model.predict(X_test)
        y_true = pd.Series(y_test).reset_index(drop=True)
        y_pred = pd.Series(preds).reset_index(drop=True)
        df_clf = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred})

        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        print(f'Accuracy: {acc:.4f}  |  Precision: {prec:.4f}  |  Recall: {rec:.4f}  |  F1: {f1:.4f}')

        print('\nClassification report:')
        print(classification_report(y_true, y_pred, zero_division=0))

        # Confusion matrix heatmap
        cm = confusion_matrix(y_true, y_pred)
        labels = sorted(pd.unique(y_true))
        cm_df = pd.DataFrame(cm, index=labels, columns=labels)
        plt.figure(figsize=(6,5))
        sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
        plt.title('Confusion Matrix')
        plt.ylabel('True')
        plt.xlabel('Predicted')
        plt.tight_layout()
        plt.show()

        print('\nSample of true vs predicted (first 10 rows):')
        display(df_clf.head(10))
else:
    print('Column `AI_Predicted_Sentiment` not found in dataframe; skipping classifier evaluation.')


Column `AI_Predicted_Sentiment` not found in dataframe; skipping classifier evaluation.


In [19]:
def load_model(name: str):
    path = MODEL_DIR / name
    if path.exists():
        return joblib.load(path)
    return None

clf = load_model('sentiment_classifier.pkl')
reg = load_model('satisfaction_regressor.pkl')

def predict_row(row: dict):
    sample = pd.DataFrame([row])
    out = {}
    if clf is not None:
        try:
            pred = clf.predict(sample)[0]
            out['AI_Predicted_Sentiment'] = str(pred)
        except Exception as e:
            out['AI_Predicted_Sentiment'] = f'Error: {e}'
    if reg is not None:
        try:
            score = float(reg.predict(sample)[0])
            out['Satisfaction_Score'] = score
        except Exception as e:
            out['Satisfaction_Score'] = f'Error: {e}'
    return out

# Example input constructed from first row for quick test (adjust columns as needed)
example_row = {c: (str(df[c].iloc[0]) if df[c].dtype == object else float(df[c].iloc[0])) for c in df.columns[:6]}
print('Example input:', example_row)
# print(predict_row(example_row))

Example input: {'Respondent_ID': 'RES_1001', 'Age_Group': '18-25', 'Gender': 'Prefer not to say', 'Ward_Number_Zone': 'Ward 21 - Bengaluru Zone', 'Survey_Topic': 'Electricity/Power', 'Citizen_Feedback_Text': 'Street light poles have exposed wires, very dangerous.'}


In [20]:
# 6.4) Save evaluation report to JSON
import json
from pathlib import Path

evaluation = {}

# Recompute metrics to fill evaluation dict safely
preproc, features, targets = prepare_pipeline(df)
X = df[features]

if 'Satisfaction_Score' in df.columns:
    y_reg = df['Satisfaction_Score']
    X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=TEST_SIZE, random_state=42)
    reg_model = load_model('satisfaction_regressor.pkl')
    if reg_model is not None:
        preds = reg_model.predict(X_test)
        evaluation['regression'] = {
            'rmse': float(mean_squared_error(y_test, preds, squared=False)),
            'mae': float(mean_absolute_error(y_test, preds)),
            'r2': float(r2_score(y_test, preds))
        }

if 'AI_Predicted_Sentiment' in df.columns:
    y_clf = df['AI_Predicted_Sentiment'].astype(str)
    X_train, X_test, y_train, y_test = train_test_split(X, y_clf, test_size=TEST_SIZE, stratify=y_clf, random_state=42)
    clf_model = load_model('sentiment_classifier.pkl')
    if clf_model is not None:
        preds = clf_model.predict(X_test)
        evaluation['classification'] = {
            'accuracy': float(accuracy_score(y_test, preds)),
            'precision': float(precision_score(y_test, preds, average='weighted', zero_division=0)),
            'recall': float(recall_score(y_test, preds, average='weighted', zero_division=0)),
            'f1': float(f1_score(y_test, preds, average='weighted', zero_division=0))
        }

out_path = MODEL_DIR / 'evaluation_report.json'
with open(out_path, 'w') as f:
    json.dump(evaluation, f, indent=2)

print('Saved evaluation report to', out_path)
print(json.dumps(evaluation, indent=2))


Saved evaluation report to /Users/kanishk/Desktop/survey analysis/model/evaluation_report.json
{}


# STEP 10: Gemini API placeholder

In [21]:
import requests
def generate_gemini_report(prompt_text: str) -> str:
    api_key = os.environ.get('GEMINI_API_KEY')
    if not api_key:
        return 'Gemini API key not set. Set GEMINI_API_KEY environment variable to enable detailed AI reports.'
    try:
        # Replace URL with your Gemini or LLM endpoint and authentication
        url = 'https://api.example.com/gemini/analyze'
        headers = {'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'}
        payload = {'prompt': prompt_text}
        resp = requests.post(url, headers=headers, json=payload, timeout=30)
        if resp.ok:
            return resp.json().get('result', resp.text)
        return f'Gemini API error: {resp.status_code} {resp.text}'
    except Exception as e:
        return f'Gemini request failed: {e}'

# Example: print(generate_gemini_report('Summarize predictions and suggest actions.'))

# STEP 12: Optional — Export Streamlit app / Save sample preview
Run this cell and set `write_app = True` to create `app.py` from the notebook. The exported app is a compact version—customize UI as needed.

In [22]:
streamlit_app_code = '''
import os
from pathlib import Path
import pandas as pd
import joblib
import streamlit as st
import plotly.express as px
import json

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
MODEL_DIR = ROOT / 'model'

def find_csv(path):
    files = list(path.glob('*.csv'))
    return files[0] if files else None

def main():
    st.set_page_config(layout='wide')
    st.title('Survey Analysis Dashboard')
    csv = find_csv(DATA_DIR)
    if not csv:
        st.warning('No CSV found in data/')
        return
    df = pd.read_csv(csv)
    st.dataframe(df.head())

if __name__ == '__main__':
    main()
'''
write_app = False  # set True to write app.py
if write_app:
    with open('app.py', 'w') as f:
        f.write(streamlit_app_code)
    print('Wrote app.py')
else:
    print('Streamlit app code ready. Set write_app = True to export app.py')

Streamlit app code ready. Set write_app = True to export app.py


In [23]:
# 6.5) Sample predictions — show model outputs on a few test rows
from IPython.display import display

preproc, features, targets = prepare_pipeline(df)
X = df[features]

print('Using TEST_SIZE =', TEST_SIZE)

# Reuse test split
if 'Satisfaction_Score' in df.columns:
    y_reg = df['Satisfaction_Score']
    X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=TEST_SIZE, random_state=42)
else:
    X_train, X_test = train_test_split(X, test_size=TEST_SIZE, random_state=42)

n_samples = min(5, len(X_test))
sample_idx = list(range(n_samples))
sample_X = X_test.iloc[sample_idx]

reg_model = load_model('satisfaction_regressor.pkl')
clf_model = load_model('sentiment_classifier.pkl')

results = sample_X.copy().reset_index(drop=True)

if reg_model is not None and 'Satisfaction_Score' in df.columns:
    try:
        results['pred_satisfaction'] = reg_model.predict(sample_X)
    except Exception as e:
        results['pred_satisfaction'] = str(e)

if clf_model is not None and 'AI_Predicted_Sentiment' in df.columns:
    try:
        results['pred_sentiment'] = clf_model.predict(sample_X)
    except Exception as e:
        results['pred_sentiment'] = str(e)

print('\nSample predictions:')
display(results)


Using TEST_SIZE = 0.2



Sample predictions:


/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.6.1 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.6.1 when using version 1.5.1. This might lead to breaking cod

,Respondent_ID,Age_Group,Gender,Ward_Number_Zone,Survey_Topic,Citizen_Feedback_Text,State,City,Area/locality,Length_of_Stay
0,RES_1096,60+,Male,Ward 25 - New Delhi Zone,Electricity/Power,Frequent power cuts in the evening without pri...,Delhi,New Delhi,Connaught Place,3
1,RES_1016,41-60,Male,Ward 24 - New Delhi Zone,Electricity/Power,"Street light poles have exposed wires, very da...",Delhi,New Delhi,Connaught Place,3
2,RES_1031,18-25,Non-binary,Ward 19 - Bengaluru Zone,Road Infrastructure,"Road construction is pending for over a month,...",Karnataka,Bengaluru,Whitefield,3
3,RES_1159,60+,Male,Ward 2 - New Delhi Zone,Waste Management,Overflowing dumpsters causing foul smell and h...,Delhi,New Delhi,Karol Bagh,3
4,RES_1129,41-60,Prefer not to say,Ward 2 - Mumbai Zone,Waste Management,Plastic waste is being dumped in the open drain.,Maharashtra,Mumbai,Andheri,3


In [24]:
# 6.6 Overall model evaluation (cross-validation + full-data fit)
# Computes cross-validated metrics and in-sample metrics after fitting on the entire dataset.
from sklearn.model_selection import cross_validate, cross_val_predict, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import numpy as np
import json

print('Starting overall model evaluation...')
preproc, features, targets = prepare_pipeline(df)
X = df[features]
overall_results = {}
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Regressor overall evaluation
if 'Satisfaction_Score' in df.columns:
    print('\n--- Regressor: 5-fold cross-validation ---')
    y_reg = df['Satisfaction_Score']
    reg = Pipeline([('preproc', preproc), ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))])
    scoring = {'neg_mse': 'neg_mean_squared_error', 'neg_mae': 'neg_mean_absolute_error', 'r2': 'r2'}
    cv_res = cross_validate(reg, X, y_reg, scoring=scoring, cv=cv, n_jobs=-1)
    mse = -cv_res['test_neg_mse']
    rmse = np.sqrt(mse)
    mae = -cv_res['test_neg_mae']
    r2s = cv_res['test_r2']
    print(f'RMSE (cv) mean: {rmse.mean():.4f}, std: {rmse.std():.4f}')
    print(f'MAE  (cv) mean: {mae.mean():.4f}, std: {mae.std():.4f}')
    print(f'R2   (cv) mean: {r2s.mean():.4f}, std: {r2s.std():.4f}')
    overall_results['regression_cv'] = {'rmse_mean': float(rmse.mean()), 'rmse_std': float(rmse.std()), 'mae_mean': float(mae.mean()), 'mae_std': float(mae.std()), 'r2_mean': float(r2s.mean()), 'r2_std': float(r2s.std())}

    # Fit on full data and evaluate in-sample
    reg.fit(X, y_reg)
    preds_full = reg.predict(X)
    rmse_full = mean_squared_error(y_reg, preds_full, squared=False)
    mae_full = mean_absolute_error(y_reg, preds_full)
    r2_full = r2_score(y_reg, preds_full)
    print('\nIn-sample metrics after fitting on full data:')
    print(f'  RMSE: {rmse_full:.4f}  |  MAE: {mae_full:.4f}  |  R2: {r2_full:.4f}')
    overall_results['regression_full'] = {'rmse': float(rmse_full), 'mae': float(mae_full), 'r2': float(r2_full)}

    # Feature importances (if available)
    try:
        rf = reg.named_steps.get('model') if hasattr(reg, 'named_steps') else reg
        fi = rf.feature_importances_
        overall_results['regression_feature_importances'] = fi.tolist()
        print('\nTop 10 feature importances:')
        importances = pd.Series(fi, index=(features if len(fi)==len(features) else [f'feat_{i}' for i in range(len(fi))]))
        display(importances.sort_values(ascending=False).head(10))
    except Exception:
        pass

# Classifier overall evaluation
if 'AI_Predicted_Sentiment' in df.columns:
    print('\n--- Classifier: 5-fold cross-validation ---')
    y_clf = df['AI_Predicted_Sentiment'].astype(str)
    clf = Pipeline([('preproc', preproc), ('model', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))])
    scoring = ['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted']
    cv_res = cross_validate(clf, X, y_clf, scoring=scoring, cv=cv, n_jobs=-1)
    print(f"Accuracy (cv) mean: {cv_res['test_accuracy'].mean():.4f}, std: {cv_res['test_accuracy'].std():.4f}")
    print(f"Precision(cv) mean: {cv_res['test_precision_weighted'].mean():.4f}, std: {cv_res['test_precision_weighted'].std():.4f}")
    print(f"Recall   (cv) mean: {cv_res['test_recall_weighted'].mean():.4f}, std: {cv_res['test_recall_weighted'].std():.4f}")
    print(f"F1       (cv) mean: {cv_res['test_f1_weighted'].mean():.4f}, std: {cv_res['test_f1_weighted'].std():.4f}")
    overall_results['classification_cv'] = {'accuracy_mean': float(cv_res['test_accuracy'].mean()), 'accuracy_std': float(cv_res['test_accuracy'].std()), 'precision_mean': float(cv_res['test_precision_weighted'].mean()), 'precision_std': float(cv_res['test_precision_weighted'].std()), 'recall_mean': float(cv_res['test_recall_weighted'].mean()), 'recall_std': float(cv_res['test_recall_weighted'].std()), 'f1_mean': float(cv_res['test_f1_weighted'].mean()), 'f1_std': float(cv_res['test_f1_weighted'].std())}

    # Cross-validated predictions for aggregated confusion matrix
    print('\nComputing cross-validated predictions for confusion matrix...')
    y_pred_cv = cross_val_predict(clf, X, y_clf, cv=cv, n_jobs=-1)
    print('Classification report (aggregated cross-val predictions):')
    print(classification_report(y_clf, y_pred_cv, zero_division=0))
    cm = confusion_matrix(y_clf, y_pred_cv)
    labels = sorted(pd.unique(y_clf))
    cm_df = pd.DataFrame(cm, index=labels, columns=labels)
    print('\nConfusion matrix (aggregated):')
    display(cm_df)
    overall_results['classification_cv_confusion'] = cm_df.to_dict()

    # Fit classifier on full data and report in-sample metrics
    clf.fit(X, y_clf)
    preds_full = clf.predict(X)
    print('\nIn-sample classification report after fitting on full data:')
    print(classification_report(y_clf, preds_full, zero_division=0))
    overall_results['classification_full'] = {'classification_report': classification_report(y_clf, preds_full, zero_division=0)}

# Save/append to evaluation_report.json
out_path = MODEL_DIR / 'evaluation_report.json'
existing = {}
if out_path.exists():
    try:
        existing = json.loads(out_path.read_text())
    except Exception:
        existing = {}
existing['overall'] = overall_results
out_path.write_text(json.dumps(existing, indent=2))
print('\nSaved overall evaluation to', out_path)
print('\nOverall evaluation summary:')
print(json.dumps(overall_results, indent=2)[:2000])


Starting overall model evaluation...

Saved overall evaluation to /Users/kanishk/Desktop/survey analysis/model/evaluation_report.json

Overall evaluation summary:
{}
